# Luantick Benchmark Example - Enhanced with Local Logic

This example notebook provides a comprehensive example of how to use the Yardstick benchmark framework to collect performance metrics from Luanti game servers and evaluate their performance under bot load. This version incorporates the local benchmark logic and supports both walkbots and blockbots.

## Running a Luanti Experiment

The cell below shows you how to run a Luanti server performance experiment using the DAS cluster. This deploys a Luanti server on one node and bots (walkbots or blockbots) on other nodes.

In [5]:
import logging
import os
import shutil
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Optional

from yardstick_benchmark.provisioning import Das
from yardstick_benchmark.monitoring import Telegraf
from yardstick_benchmark.games.luanti.server import LuantiServer
from yardstick_benchmark.games.luanti.workload import RustWalkAround, RustBlockBot
import yardstick_benchmark

from time import sleep
import tempfile

# Configure logging for better visibility
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

print("✓ Imports successful")

def check_dependencies():
    """Check if required tools and paths are available."""
    logger.info("Checking dependencies...")
    
    # Check if yardstick_benchmark is properly installed
    try:
        import yardstick_benchmark
        logger.info("✓ yardstick_benchmark module available")
    except ImportError:
        logger.error("✗ yardstick_benchmark module not found")
        raise ImportError("Please ensure the yardstick benchmark framework is properly installed")
    
    # Check if bot components exist
    bot_dir = Path("bot_components/texmodbot")
    if not bot_dir.exists():
        logger.error(f"✗ Rust bot directory not found: {bot_dir}")
        raise FileNotFoundError("Please ensure bot_components/texmodbot exists")
    logger.info(f"✓ Rust bot components found: {bot_dir}")
    
    # Check output directory permissions
    dest = Path(f"/var/scratch/{os.getlogin()}/yardstick/luanti_output")
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        logger.info(f"✓ Output directory accessible: {dest}")
    except Exception as e:
        logger.error(f"✗ Cannot access output directory: {e}")
        raise
    
    return dest

# Run dependency check
dest = check_dependencies()
print(f"Dependencies checked successfully. Results will be saved to: {dest}")

# 🎮 LUANTI BENCHMARK CONFIGURATION - UPDATED PROVEN BUILD METHOD
# ================================================================
# This configuration uses our tested and proven build method that works on DAS5

# === MAIN SETTINGS ===
BOTS_PER_NODE = 10          # Number of bots per bot node (tested and working)
BENCHMARK_DURATION = 120    # Benchmark duration in seconds
NUM_NODES = 2               # Start with 2 nodes: 1 server + 1 bot node

# === BOT TYPE SETTINGS ===
BOT_TYPE = "walkbot"        # Use walkbot (proven to work)
MOVEMENT_MODE = "random"    # Random movement (tested)
MOVEMENT_SPEED = 2.0        # Speed in seconds between actions

# === GAME CONFIGURATION ===
GAME_MODE = "minetest_game" # Use standard minetest_game (most reliable)

# === PROVEN BUILD CONFIGURATION ===
# Based on our successful manual testing and documentation
USE_HEADLESS_BUILD = True           # Use headless server build (no client dependencies)
BUILD_WITH_LUAJIT = True            # Build with LuaJIT for better performance
ENABLE_IPV4_ONLY = True             # Use IPv4 only (fixes connection issues)
USE_SYSTEM_LIBS = True              # Use system libraries where available
DISABLE_UNNECESSARY_FEATURES = True  # Disable gettext, client features, etc.

# === NETWORK CONFIGURATION ===
SERVER_PORT = 30000         # Standard Luanti port
SERVER_BIND_ADDRESS = "127.0.0.1"  # IPv4 localhost binding
DISABLE_IPV6 = True         # Explicitly disable IPv6

# === SPAWN AREA POSITIONING ===
SPAWN_X = 0
SPAWN_Y = 9.5
SPAWN_Z = 123
BUILD_NEAR_SPAWN = True     # Position bots near spawn area

# === ADVANCED SETTINGS ===
COLLECT_ALL_NODES = True    # Monitor all nodes (server + all bot nodes)
VERBOSE_PROGRESS = True     # Show detailed progress during benchmark

# === CALCULATED VALUES ===
TOTAL_BOTS = BOTS_PER_NODE * (NUM_NODES - 1)  # Total bots across all bot nodes


for i in range(2, NUM_NODES + 1):
    bot_group = chr(64 + i - 1)  # A, B, C, etc.


expected_runtime = BENCHMARK_DURATION + 300  # 5 minutes overhead for build

2025-07-28 22:57:19 - INFO - Checking dependencies...
2025-07-28 22:57:19 - INFO - ✓ yardstick_benchmark module available
2025-07-28 22:57:19 - INFO - ✓ Rust bot components found: bot_components/texmodbot
2025-07-28 22:57:19 - INFO - ✓ Output directory accessible: /var/scratch/aco237/yardstick/luanti_output


✓ Imports successful
Dependencies checked successfully. Results will be saved to: /var/scratch/aco237/yardstick/luanti_output


## Provision DAS nodes

In [6]:
def provision_nodes_with_validation(num_nodes: int = 2):
    """Provision nodes on the DAS cluster with validation."""
    logger.info(f"Provisioning {num_nodes} nodes on DAS cluster...")
    
    das = Das()
    try:
        nodes = das.provision(num=num_nodes, time_s=1500)
        
        logger.info(f"✓ Successfully provisioned {len(nodes)} nodes:")
        for i, node in enumerate(nodes):
            logger.info(f"  Node {i}: {node.host} (wd: {node.wd})")
        return das, nodes
    except Exception as e:
        logger.error(f"✗ Failed to provision nodes: {e}")
        raise

start_time = datetime.now()


das, nodes = provision_nodes_with_validation(num_nodes=NUM_NODES)

# Remove previous results if they exist
if dest.exists():
    print(f"Removing previous results at {dest}")
    shutil.rmtree(dest)

# Clean any previous data on nodes
yardstick_benchmark.clean(nodes)

2025-07-28 22:57:21 - INFO - Provisioning 2 nodes on DAS cluster...
2025-07-28 22:57:22 - INFO - ✓ Successfully provisioned 2 nodes:
2025-07-28 22:57:22 - INFO -   Node 0: node005 (wd: /local/aco237/yardstick/node005)
2025-07-28 22:57:22 - INFO -   Node 1: node006 (wd: /local/aco237/yardstick/node006)



PLAY [Clean data from nodes] ***************************************************

TASK [Gathering Facts] *********************************************************
ok: [node005]
ok: [node006]

TASK [Remove data from nodes] **************************************************
ok: [node006]
ok: [node005]

PLAY RECAP *********************************************************************
node005                    : ok=2    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
node006                    : ok=2    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


## Start Telegraf for metrics

In [ ]:
telegraf = Telegraf(nodes)
    
telegraf.add_input_luanti_metrics(nodes[0])  # Server node
res = telegraf.deploy()
telegraf.start()


## Deploy Luanti headless server

In [7]:
luanti_server = LuantiServer(
    nodes[:1], 
    game_mode=GAME_MODE,        # Use our configured game mode
    use_source_build=False,     # Use source build method (for headless)
    enable_luajit=False,        # Enable LuaJIT
    ipv4_only=ENABLE_IPV4_ONLY  # IPv4 only configuration
)

# Debug: Check the script paths
print(f"Deploy script path: {luanti_server.deploy_action.script}")
print(f"Script exists: {luanti_server.deploy_action.script.exists()}")
print(f"Script is file: {luanti_server.deploy_action.script.is_file()}")
print(f"Current working directory: {os.getcwd()}")


try:
    luanti_server.deploy()
    print("✅ Luanti server deployed successfully")
    
    print("🚀 Starting Luanti headless server...")
    luanti_server.start()
    print("✅ Luanti headless server started successfully")
    
    # Give server time to fully initialize
    print("⏳ Allowing server to initialize (10 seconds)...")
    sleep(10)
    
except Exception as e:
    print(f"❌ Error starting Luanti server: {e}")
    print("Server may have failed to bind to the configured address/port.")
    raise

Deploy script path: /var/scratch/aco237/luantick/yardstick_benchmark/games/luanti/server/luanti_deploy.yml
Script exists: True
Script is file: True
Current working directory: /var/scratch/aco237/luantick

PLAY [Deploy Pre-compiled Luanti Server] ***************************************

TASK [Gathering Facts] *********************************************************
ok: [node005]

TASK [Create working directory and subdirectories] *****************************
changed: [node005] => (item=/local/aco237/yardstick/node005/luanti_server-yg7qmcxt)
changed: [node005] => (item=/local/aco237/yardstick/node005/luanti_server-yg7qmcxt/lib)
changed: [node005] => (item=/local/aco237/yardstick/node005/luanti_server-yg7qmcxt/logs)
changed: [node005] => (item=/local/aco237/yardstick/node005/luanti_server-yg7qmcxt/games)
changed: [node005] => (item=/local/aco237/yardstick/node005/luanti_server-yg7qmcxt/builtin)
changed: [node005] => (item=/local/aco237/yardstick/node005/luanti_server-yg7qmcxt/worlds/ben

## Start Luanti server

In [ ]:
try:
    print("🚀 Starting Luanti headless server...")
    luanti_server.start()
    print("✅ Luanti headless server started successfully")
    
    # Give server time to fully initialize
    print("⏳ Allowing server to initialize (10 seconds)...")
    sleep(10)
    
except Exception as e:
    print(f"❌ Error starting Luanti server: {e}")
    print("Server may have failed to bind to the configured address/port.")
    raise

## Run Walkbot Benchmark

In [ ]:
# Start walkbot benchmark with 20 bots for 3 minutes
import subprocess
import time
from datetime import datetime

def start_walkbot_benchmark():
    """Start walkbot benchmark with 20 bots for 3 minutes."""
    node = nodes[0]
    
    # Find the current server directory from the framework
    server_dirs = []
    list_cmd = f"ssh {node.host} 'find {node.wd} -name \"luanti_server-*\" -type d 2>/dev/null'"
    result = subprocess.run(list_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        server_dirs = result.stdout.strip().split('\n')
    
    if not server_dirs:
        print("❌ No Luanti server directory found!")
        print(f"🔍 Searching in: {node.wd}")
        # Debug: list all directories
        debug_cmd = f"ssh {node.host} 'ls -la {node.wd}'"
        debug_result = subprocess.run(debug_cmd, shell=True, capture_output=True, text=True)
        print(f"Available directories: {debug_result.stdout}")
        return False
    
    server_dir = server_dirs[0]  # Use the first/most recent one
    print(f"Using server directory: {server_dir}")
    
    # Check if server is actually running
    check_cmd = f"ssh {node.host} 'ps aux | grep luantiserver | grep -v grep | grep \"port 30000\"'"
    result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
    
    # Change this line from checking "running" to checking if any output exists
    if not result.stdout.strip():  # FIXED: Check for any process output instead of "running"
        print("❌ Server is not running! Checking server logs...")
        log_cmd = f"ssh {node.host} 'cd {server_dir} && cat logs/startup.log 2>/dev/null || echo \"No startup log\"'"
        log_result = subprocess.run(log_cmd, shell=True, capture_output=True, text=True)
        print(f"Server startup log: {log_result.stdout}")
        return False
    
    print("✅ Server is running, starting walkbot benchmark...")
    
    # Bot configuration
    NUM_BOTS = 20
    DURATION_SECONDS = 180  # 3 minutes
    SERVER_ADDRESS = f"{node.host}:30000"
    
    print(f"🤖 Starting {NUM_BOTS} walkbots for {DURATION_SECONDS} seconds")
    print(f"📡 Connecting to server at: {SERVER_ADDRESS}")
    print(f"🕒 Start time: {datetime.now().strftime('%H:%M:%S')}")
    
    # FIXED: Use timedelta instead of int for duration
    try:
        walkbot_workload = RustWalkAround(
            nodes[1:] if len(nodes) > 1 else nodes[:1],  # Use second node if available
            server_host=node.host,
            server_port=30000,
            bots_per_node=NUM_BOTS,
            duration=timedelta(seconds=DURATION_SECONDS),  # FIXED: Use timedelta
            movement_mode="random",
            movement_speed=2.0
        )
        
        print("🚀 Deploying walkbots...")
        walkbot_workload.deploy()
        
        print("🏃 Starting walkbot workload...")
        walkbot_workload.start()
        
        print(f"⏳ Running benchmark for {DURATION_SECONDS + 10} seconds...")
        print("📊 Monitor server metrics during this time...")
        
        # Wait for the benchmark to complete
        time.sleep(DURATION_SECONDS + 10)  # Extra 10 seconds for cleanup
        
        print("🏁 Walkbot benchmark completed!")
        
        # Check final server status and metrics
        check_metrics_cmd = f"""
        ssh {node.host} 'cd {server_dir} && 
        echo "=== Server Status ===" &&
        if [ -f luantiserver.pid ]; then PID=$(cat luantiserver.pid); if ps -p $PID > /dev/null; then echo "✅ Server still running (PID: $PID)"; else echo "❌ Server stopped"; fi; else echo "❌ No PID file"; fi &&
        echo "=== Recent Server Logs ===" &&
        tail -10 logs/server.log 2>/dev/null || echo "No server logs" &&
        echo "=== World Metrics Files ===" &&
        ls -la worlds/benchmark/mod_storage/ 2>/dev/null || echo "No mod_storage directory"'
        """
        
        result = subprocess.run(check_metrics_cmd, shell=True, capture_output=True, text=True)
        print("📋 Post-benchmark status:")
        print(result.stdout)
        
        return True
        
    except Exception as e:
        print(f"❌ Error running walkbot benchmark: {e}")
        print("🔧 Trying manual bot deployment...")
        
        # Fallback: manual bot deployment if framework fails
        # return start_manual_walkbots(node, server_dir, NUM_BOTS, DURATION_SECONDS)

# def start_manual_walkbots(node, server_dir, num_bots, duration):
#     """Fallback method to start walkbots manually."""
#     print("🔧 Starting walkbots manually...")
    
#     # Check if we have the bot binary
#     bot_check_cmd = "which texmodbot 2>/dev/null || find /var/scratch/aco237/luantick -name 'texmodbot' -type f 2>/dev/null | head -1"
#     result = subprocess.run(bot_check_cmd, shell=True, capture_output=True, text=True)
    
#     if not result.stdout.strip():
#         print("❌ No walkbot binary found!")
#         print("💡 You may need to build the bots first or check the bot_components directory")
#         return False
    
#     bot_binary = result.stdout.strip()
#     print(f"🤖 Using bot binary: {bot_binary}")
    
#     # Start bots one by one
#     bot_processes = []
#     for i in range(num_bots):
#         username = f"walkbot_{i:03d}"
        
#         # Start bot in background
#         bot_cmd = f"""
#         {bot_binary} {node.host}:30000 \\
#         --username {username} \\
#         --password benchmark123 \\
#         --auto-register \\
#         --quit-after-seconds {duration} \\
#         --mode random \\
#         --speed 2.0 &
#         """
        
#         subprocess.Popen(bot_cmd, shell=True)
#         time.sleep(0.5)  # Stagger connections
        
#         if (i + 1) % 5 == 0:  # Progress update every 5 bots
#             print(f"🤖 Started {i + 1}/{num_bots} bots...")
    
#     print(f"🏃 All {num_bots} walkbots started!")
#     print(f"⏳ Running for {duration} seconds...")
    
#     time.sleep(duration + 10)
#     print("🏁 Manual walkbot benchmark completed!")
    
#     return True

# Run the benchmark
print("🎮 WALKBOT BENCHMARK - 20 BOTS, 3 MINUTES")
print("=" * 50)

if start_walkbot_benchmark():
    print("✅ Walkbot benchmark completed successfully!")
    print("📊 Check the metrics files in the server directory for performance data")
else:
    print("❌ Walkbot benchmark failed")
    print("🔍 Check server logs and connectivity")

print("=" * 50)

🎮 WALKBOT BENCHMARK - 20 BOTS, 3 MINUTES
❌ No Luanti server directory found!
🔍 Searching in: /local/aco237/yardstick/node005
Available directories: total 4
drwxr-xr-x 5 aco237 aco237  112 Jul 28 23:03 .
drwxr-xr-x 3 aco237 aco237   28 Jul 28 22:58 ..
lrwxrwxrwx 1 aco237 aco237   62 Jul 28 23:02 builtin -> /local/aco237/yardstick/node005/luanti_server-yg7qmcxt/builtin
drwxrwxr-x 2 aco237 aco237   33 Jul 28 23:03 cache
lrwxrwxrwx 1 aco237 aco237   60 Jul 28 23:02 games -> /local/aco237/yardstick/node005/luanti_server-yg7qmcxt/games
drwxr-xr-x 7 aco237 aco237 4096 Jul 28 23:03 luanti_server-yg7qmcxt
drwxrwxr-x 2 aco237 aco237   10 Jul 28 23:03 mod_data

❌ Walkbot benchmark failed
🔍 Check server logs and connectivity


## Run Blockbot benchmark

In [ ]:
# Start blockbot benchmark with building bots
import subprocess
import time
from datetime import datetime, timedelta

def start_blockbot_benchmark():
    """Start blockbot benchmark with building bots."""
    node = nodes[0]
    
    # Find the current server directory
    server_dirs = []
    list_cmd = f"ssh {node.host} 'find {node.wd} -name \"luanti_server-*\" -type d 2>/dev/null'"
    result = subprocess.run(list_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        server_dirs = result.stdout.strip().split('\n')
    
    if not server_dirs:
        print("❌ No Luanti server directory found!")
        return False
    
    server_dir = server_dirs[0]
    print(f"Using server directory: {server_dir}")
    
    # Check if server is running
    check_cmd = f"ssh {node.host} 'ps aux | grep luantiserver | grep -v grep | grep \"port 30000\"'"
    result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
    
    if not result.stdout.strip():
        print("❌ Server is not running!")
        return False
    
    print("✅ Server is running, starting blockbot benchmark...")
    
    # Bot configuration
    NUM_BOTS = 10
    DURATION_SECONDS = 180  # 3 minutes
    SERVER_ADDRESS = f"{node.host}:30000"
    
    print(f"🏗️ Starting {NUM_BOTS} blockbots for {DURATION_SECONDS} seconds")
    print(f"📡 Connecting to server at: {SERVER_ADDRESS}")
    print(f"🕒 Start time: {datetime.now().strftime('%H:%M:%S')}")
    
    try:
        blockbot_workload = RustBlockBot(
            nodes[1:] if len(nodes) > 1 else nodes[:1],  # Use second node if available
            server_host=node.host,
            server_port=30000,
            bots_per_node=NUM_BOTS,
            duration=timedelta(seconds=DURATION_SECONDS),
            building_pattern="tower",    # Try tower pattern
            building_speed=2.0,          # 2 seconds between blocks
            max_blocks=50,               # Limit blocks per bot
            destructive_mode=False,      # Don't dig blocks initially
            start_x=10.0,                # Build away from spawn
            start_y=8.0,                 # Ground level
            start_z=130.0                # Near spawn Z coordinate
        )
        
        print("🏗️ Deploying blockbots...")
        blockbot_workload.deploy()
        
        print("🏃 Starting blockbot workload...")
        blockbot_workload.start()
        
        print(f"⏳ Running benchmark for {DURATION_SECONDS + 10} seconds...")
        print("📊 Monitor server metrics during this time...")
        
        # Wait for the benchmark to complete
        time.sleep(DURATION_SECONDS + 10)
        
        print("🏁 Blockbot benchmark completed!")
        
        # Check final server status
        check_metrics_cmd = f"""
        ssh {node.host} 'cd {server_dir} && 
        echo "=== Server Status ===" &&
        if [ -f luantiserver.pid ]; then PID=$(cat luantiserver.pid); if ps -p $PID > /dev/null; then echo "✅ Server still running (PID: $PID)"; else echo "❌ Server stopped"; fi; else echo "❌ No PID file"; fi &&
        echo "=== Recent Server Logs ===" &&
        tail -10 logs/startup.log | grep -E "(YARDSTICK|blockbot)" || echo "No blockbot activity in logs"'
        """
        
        result = subprocess.run(check_metrics_cmd, shell=True, capture_output=True, text=True)
        print("📋 Post-benchmark status:")
        print(result.stdout)
        
        return True
        
    except Exception as e:
        print(f"❌ Error running blockbot benchmark: {e}")
        return False

# Run the benchmark
print("🏗️ BLOCKBOT BENCHMARK - 10 BUILDING BOTS, 3 MINUTES")
print("=" * 50)

if start_blockbot_benchmark():
    print("✅ Blockbot benchmark completed successfully!")
    print("📊 Check the metrics files for building performance data")
else:
    print("❌ Blockbot benchmark failed")

print("=" * 50)

🏗️ BLOCKBOT BENCHMARK - 10 BUILDING BOTS, 3 MINUTES
Using server directory: /local/aco237/yardstick/node005/luanti_server-yg7qmcxt
✅ Server is running, starting blockbot benchmark...
🏗️ Starting 10 blockbots for 180 seconds
📡 Connecting to server at: node005:30000
🕒 Start time: 23:08:09
🏗️ Deploying blockbots...

PLAY [Deploy Rust BlockBot bots] ***********************************************

TASK [Gathering Facts] *********************************************************


## Analyze metrics luanti

In [ ]:
def analyze_extracted_metrics():
    """Analyze the downloaded performance metrics."""
    import pandas as pd
    import matplotlib.pyplot as plt
    
    # Load the tick metrics
    tick_file = Path("./extracted_metrics/tick_metrics_extracted.tsv")
    player_file = Path("./extracted_metrics/player_events_extracted.tsv")
    
    if tick_file.exists():
        # First, let's debug what's in the file
        print("🔍 Debugging TSV file structure:")
        with open(tick_file, 'r') as f:
            first_few_lines = f.readlines()[:5]
            for i, line in enumerate(first_few_lines):
                print(f"Line {i}: {repr(line)}")
        
        # Try to read with different separators
        try:
            # First try with tab separator
            df_ticks = pd.read_csv(tick_file, sep='\t')
            print(f"✅ Read with tab separator. Columns: {list(df_ticks.columns)}")
        except:
            try:
                # If that fails, try with space separator (in case sed didn't work properly)
                df_ticks = pd.read_csv(tick_file, sep=' ', header=0)
                print(f"✅ Read with space separator. Columns: {list(df_ticks.columns)}")
            except Exception as e:
                print(f"❌ Error reading file: {e}")
                return
        
        # Fix column names if they have escaped tabs
        if len(df_ticks.columns) == 1 and '\\t' in df_ticks.columns[0]:
            print("🔧 Fixing escaped tab characters in header...")
            # The header has escaped tabs, need to fix this
            header_line = first_few_lines[0].replace('\\t', '\t').strip()
            columns = header_line.split('\t')
            
            # Re-read the file, skipping the bad header
            df_ticks = pd.read_csv(tick_file, sep='\t', skiprows=1, names=columns)
            print(f"✅ Fixed columns: {list(df_ticks.columns)}")
        
        print("\n📊 Data sample:")
        print(df_ticks.head())
        print(f"\nData types:\n{df_ticks.dtypes}")
        
        print("\n📈 PERFORMANCE ANALYSIS:")
        print("=" * 50)
        
        # Basic statistics
        baseline = df_ticks[df_ticks['players'] == 0]['step_ms']
        under_load = df_ticks[df_ticks['players'] == 20]['step_ms']
        
        print(f"📊 Baseline Performance (0 players):")
        print(f"   • Average step time: {baseline.mean():.2f}ms")
        print(f"   • Min/Max: {baseline.min():.2f}ms / {baseline.max():.2f}ms")
        
        if len(under_load) > 0:
            print(f"\n🤖 Under Load (20 players):")
            print(f"   • Average step time: {under_load.mean():.2f}ms")
            print(f"   • Min/Max: {under_load.min():.2f}ms / {under_load.max():.2f}ms")
            print(f"   • Performance degradation: {((under_load.mean() - baseline.mean()) / baseline.mean() * 100):.1f}%")
        else:
            print(f"\n⚠ No data found for 20 players - checking available player counts:")
            print(f"Available player counts: {sorted(df_ticks['players'].unique())}")
        
        # Player count changes
        player_changes = df_ticks['players'].diff().fillna(0)
        load_events = len(player_changes[player_changes != 0])
        
        print(f"\n📋 Test Summary:")
        print(f"   • Total measurement duration: {df_ticks['uptime_s'].max():.1f} seconds")
        print(f"   • Data points collected: {len(df_ticks)} ticks")
        print(f"   • Player load changes: {load_events} events")
        print(f"   • Peak concurrent players: {df_ticks['players'].max()}")
        
        # Quick visualization
        plt.figure(figsize=(12, 8))
        
        plt.subplot(2, 1, 1)
        plt.plot(df_ticks['uptime_s'], df_ticks['step_ms'], 'b-', alpha=0.7, linewidth=1)
        plt.axhline(y=50, color='r', linestyle='--', label='Target (50ms)', alpha=0.8)
        plt.ylabel('Step Time (ms)')
        plt.title('Server Performance Over Time')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(2, 1, 2)
        plt.plot(df_ticks['uptime_s'], df_ticks['players'], 'g-', linewidth=2)
        plt.ylabel('Connected Players')
        plt.xlabel('Uptime (seconds)')
        plt.title('Player Count Over Time')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('./extracted_metrics/performance_analysis.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"\n📁 Performance chart saved: ./extracted_metrics/performance_analysis.png")
        
    else:
        print("❌ Tick metrics file not found!")
        return
        
    if player_file.exists():
        with open(player_file, 'r') as f:
            events = f.readlines()
        
        joins = [line for line in events if 'joined:' in line]
        leaves = [line for line in events if 'left:' in line]
        
        print(f"\n🤖 Player Events:")
        print(f"   • Join events: {len(joins)}")
        print(f"   • Leave events: {len(leaves)}")
        print(f"   • Event balance: {'✅ Balanced' if len(joins) == len(leaves) else '⚠ Imbalanced'}")
    else:
        print("⚠ Player events file not found!")

# Run the analysis
analyze_extracted_metrics()

## Clean up nodes

In [ ]:
yardstick_benchmark.clean(nodes)
das.release(nodes)